In [10]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

In [11]:
# MovieLens files use '::' as separator

movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding='latin1'
)

ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding='latin1'
)

users = pd.read_csv(
    "users.dat",
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"],
    encoding='latin1'
)

# Merge all tables
data = ratings.merge(movies, on="movie_id").merge(users, on="user_id")

In [12]:
# 3. BASIC DATA CHECKS


print("Users:", data.user_id.nunique())
print("Movies:", data.movie_id.nunique())
print("Ratings:", len(data))


Users: 6040
Movies: 3706
Ratings: 1000209


In [13]:
# 4. POPULARITY BASED RECOMMENDER
# Used as a baseline and for cold-start users

movie_stats = data.groupby("movie_id").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

movie_stats["popularity_score"] = (
    movie_stats["avg_rating"] * movie_stats["rating_count"]
)

popular_movies = movie_stats.sort_values(
    "popularity_score", ascending=False
).merge(movies, on="movie_id")


In [14]:
# 5. CONTENT BASED RECOMMENDATION
# Using movie genres only (easy to explain)

tfidf = TfidfVectorizer(tokenizer=lambda x: x.split('|'))
tfidf_matrix = tfidf.fit_transform(movies["genres"])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

movie_index = pd.Series(movies.index, index=movies.movie_id)

def content_based_recommend(movie_id, top_n=10):
    idx = movie_index[movie_id]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_movies = [i[0] for i in scores[1:top_n+1]]
    return movies.iloc[top_movies][["movie_id", "title", "genres"]]


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [15]:
# 6. ITEM BASED COLLABORATIVE FILTERING
# More stable than user-based CF

item_user_matrix = data.pivot_table(
    index="movie_id",
    columns="user_id",
    values="rating"
).fillna(0)

item_similarity = cosine_similarity(item_user_matrix)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

def item_based_recommend(movie_id, top_n=10):
    similar_items = item_similarity_df[movie_id].sort_values(ascending=False)
    return similar_items.iloc[1:top_n+1].index.tolist()



In [16]:
# 7. CANDIDATE GENERATION (RECALL)
# Reduce search space before ranking

def generate_candidates(user_id, max_candidates=50):
    watched = data[data.user_id == user_id]["movie_id"].tolist()
    candidates = set()

    for m in watched[:5]:
        candidates.update(item_based_recommend(m, top_n=10))

    candidates.update(popular_movies.head(20)["movie_id"].tolist())

    return list(candidates)[:max_candidates]


In [17]:
# 8. FEATURE ENGINEERING FOR RANKING
# Simple and interpretable features

def create_features(user_id, movie_id):
    user_avg = data[data.user_id == user_id]["rating"].mean()
    movie_avg = data[data.movie_id == movie_id]["rating"].mean()
    popularity = movie_stats[movie_stats.movie_id == movie_id]["rating_count"].values[0]
    return [user_avg, movie_avg, popularity]

In [ ]:
# 9. TRAIN RANKING MODEL
# Tree-based model

X = []
y = []

for _, row in data.sample(40000, random_state=42).iterrows():
    X.append(create_features(row.user_id, row.movie_id))
    y.append(row.rating)

X = np.array(X)
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rank_model = GradientBoostingRegressor()
rank_model.fit(X_train, y_train)


In [ ]:
# 10. FINAL HYBRID RECOMMENDER
# Combines CF + popularity + ranking

def recommend_movies(user_id, top_n=10):

    if user_id not in data.user_id.values:
        return popular_movies.head(top_n)[["movie_id", "title"]]

    candidates = generate_candidates(user_id)
    scored = []

    for movie_id in candidates:
        features = create_features(user_id, movie_id)
        score = rank_model.predict([features])[0]
        scored.append((movie_id, score))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    top_movies = [m[0] for m in scored[:top_n]]

    return movies[movies.movie_id.isin(top_movies)][["movie_id", "title"]]


In [ ]:
# 11. EVALUATION
# RMSE used only as a sanity check

preds = rank_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print("RMSE:", rmse)


# ===============================
# 12. SIMPLE EXPLAINABILITY
# ===============================
# Helps answer "why this recommendation?"

def explain_recommendation(user_id, movie_id):
    return {
        "reason": "Recommended based on similar users, item popularity, and your rating history",
        "user_avg_rating": data[data.user_id == user_id]["rating"].mean(),
        "movie_avg_rating": data[data.movie_id == movie_id]["rating"].mean()
    }
